# Algebraic Reduction and Numerical Optimization Methods for solving Quantum Control Sum of Squares Problems

This research is a work in progress. You can find the preprint IEEE paper at https://alexanderleong.com/public/qcsos_paper.pdf. This notebook is meant to complement the details described in the paper. As the solver continues to improve you should expect the quality of the results to vary. If you have problems running the package, notebook or if you find errors, contact the author directly by email at toshibaalexander@gmail.com.

**IMPORTANT NOTES** \
You must run with Julia 1.11 (tested on Google Colab Julia Runtime Version 2025-10). Later versions yet to be supported. \
The current Julia package ConicSolve.jl is out of date. You must use the latest version in the GitHub repo.

## Background

This article explores algebraic methods for quantum control polynomial optimization problems. We show how degeneracy, numerical stability and scalability are central issues in quantum control and how ConicSolve.jl has been extended to use face reduction and symmetry reduction (via the Wedderburn decomposition).

Many quantum control problems are posed as polynomial optimization problems. Sum of Squares programming is effective at utilizing symmetric self-dual solvers due to such solvers exhibiting polynomial-time convergence and achieving solutions to high precision. Despite this solving Semidefinite programs and moreover Sum of Squares programs remain challenging. First is the computational blowup in memory and runtime cost as the problem size increases, second is the ill conditioning of solving polynomial optimization problems using the monomial basis, third is the numerical struggle that solvers must deal with.

Sum of squares programming is widely used in control (Lyapunov stability), robotics (trajectory optimization) and the validation and verification of safety critical systems. Global convergence and solution quality are becoming increasingly relevant in quantum control and quantum information sciences. Scaling quantum systems without compromising solution quality remains an ongoing challenge. This motivates the question; how can we adapt existing well established algebraic and computational tools in the quantum realm.

In this article we’ll focus on the fixed-time control problem of finding the control that achieves as close as possible a given target unitary at the end of a given evolution time. The QCPOP method popularized by Bondar et al., 2025 uses TSSOS.jl for solving the sum of squares programming problem. We take a different approach by proposing a computational pipeline based on the ideas discussed in Permenter, 2017 to apply face reduction on top of symmetry reduction as a preprocessing step before solving the SOS problem. This is attractive for several theoretical/practical reasons, it exploits problem structure without relaxing the original problem, these mathematical methods can act as additional layers to extend existing solvers, packages such as SymbolicWedderburn.jl (which we use) already exists for exploiting algebraic structure.

An ongoing challenge with adopting these tools is a robust solver architecture. We’ll show the changes in ConicSolve.jl made to further reduce degeneracy and numerical ill-conditioning, particularly the use of matrix decompositions, equilibration and regularization methods. We hope this article stimulates discussion and interest in further advancing Julia’s ecosystem to realize and explore new algebraic and computational methods. It is without many Julia packages like OperatorScaling.jl and SymbolicWedderburn.jl that this article and exploration of these ideas would remain just that, a theoretical ideal.

To get started with QCSOS.jl we need to import a few packages. **This will take awhile ~60 mins in Google Colab. We recommend running this notebook on your local machine!**

In [1]:
run(`git clone -b solver_extensions https://github.com/alexander-leong/ConicSolve.jl.git`)
run(`git clone -b test https://github.com/alexander-leong/QCSOS.jl.git`)

if Int64(VERSION.major) != 1 || Int64(VERSION.minor) != 11
    @warn "Unsupported Julia major/minor version!"
else
    using Pkg
    Pkg.add(path="/content/ConicSolve.jl")
    Pkg.develop(path="/content/ConicSolve.jl/ext/experimental/ConicSolveFR.jl")
    Pkg.add(path="/content/QCSOS.jl")
    Pkg.activate("/content/QCSOS.jl")
end

Cloning into 'ConicSolve.jl'...
Cloning into 'QCSOS.jl'...
     Cloning git-repo `/content/ConicSolve.jl`
    Updating git-repo `/content/ConicSolve.jl`
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed Enzyme_jll ────────────────── v0.0.303+0
   Installed Libmount_jll ──────────────── v2.42.4+0
   Installed GR_jll ────────────────────── v0.73.27+0
   Installed JpegTurbo_jll ─────────────── v3.2.0+1
   Installed LRUCache ──────────────────── v1.6.2
   Installed ImageIO ───────────────────── v0.6.10
   Installed MutableArithmetics ────────── v1.9.0
   Installed TiffImages ────────────────── v0.11.9
   Installed LERC_jll ──────────────────── v4.2.0+0
   Installed HypergeometricFunctions ───── v0.3.30
   Installed SciMLPublic ───────────────── v1.3.0
   Installed PNGFiles ──────────────────── v0.4.5
   Installed Preferences ───────────────── v1.6.1
   Installed Accessors ─────────────────── v0.1.45
   Installed NNlib ────────────────

In [2]:
using Pkg
Pkg.add("DynamicPolynomials")
using DynamicPolynomials
using LinearAlgebra
using ConicSolve
using QCSOS
using Random

   Resolving package versions...
   Installed StatsFuns ───────────────── v1.5.2
   Installed MutableArithmetics ──────── v1.6.7
   Installed WGLMakie ────────────────── v0.13.7
   Installed SciMLLogging ────────────── v1.4.0
   Installed Unitful ─────────────────── v1.25.1
   Installed Accessors ───────────────── v0.1.42
   Installed FastGaussQuadrature ─────── v1.1.0
   Installed NonlinearSolve ──────────── v4.12.0
   Installed SciMLJacobianOperators ──── v0.1.11
   Installed MaybeInplace ────────────── v0.1.4
   Installed PDMats ──────────────────── v0.11.36
   Installed StaticArrays ────────────── v1.9.18
   Installed Polyester ───────────────── v0.7.18
   Installed PermutationGroups ───────── v0.6.3
   Installed PlotUtils ───────────────── v1.4.4
   Installed RecursiveArrayTools ─────── v3.39.0
   Installed OpenSSL ─────────────────── v1.6.0
   Installed ZygoteRules ─────────────── v0.2.7
   Installed DataFrames ──────────────── v1.8.1
   Installed Static ──────────────────── v1.3

## The fixed time control problem

The fixed time control problem involves finding the control coefficients in ``E(t)`` that results in a unitary ``U(T)`` which is as close as possible to a given target unitary, ``U*`` at time ``T``.

Specifically we solve the optimization problem
$$ \begin{aligned}\text{minimize}\qquad & Tr(M^{\dagger}M) \\
\text{subject to}\qquad & \partial_tU(t) = A(t)U(t) \\
\qquad & U(0) = \mathbb{1} \\
\qquad & A(t) = -i(H_0 + E(t)V) \end{aligned} $$

where the objective function in Eq. (1) is approximated as
$$ \|e^{\Lambda_n/2} - e^{-\Lambda_n/2}U^*\|^2 $$

and the control function takes the form
$$ E(t) = \Sigma_{k=0}^{m-1}x_kt^k $$

where $x_k$ are the coefficients being solved for.

Below we define the problem variables.

In [3]:
i = 1
T = 0.5
# Drift Hamiltonian
H0 = [
    0 0 0;
    0 3.21505101e+10 0;
    0 0 6.23173079e+10
];

H0 ./= norm(H0, Inf);

# Control Hamiltonian
V = [
    0 1 0;
    1 0 1.41421356;
    0 1.41421356 0
];

V ./= norm(V, Inf);

deg = 4
n_samples = 1000
Random.seed!(6292022)
problem = QuantumControlSOSProblem(H0, V, T, deg)
exact_x = -1 .+ 2 * rand(length(problem.x) * n_samples)
exact_x = reshape(exact_x, (length(problem.x), n_samples))
U_target = get_unitary(problem, exact_x[:, i])

3×3 Matrix{ComplexF64}:
   0.975721+0.00204083im  0.0281255+0.214463im  -0.0331948+0.00880109im
  0.0266327+0.214653im     0.896927-0.234773im    0.114424+0.283769im
 -0.0333103+0.00835158im   0.112561+0.284513im    0.833099-0.459509im

We can approximate the solution to the initial value problem of the ODE
$$ \partial_tU(t) = A(t)U(t), U(0) = 1 $$
using the Magnus expansion, for the first three terms this is \
$ \Omega_1(t) = \int_0^t A(t_1) dt_1 $, \
$ \Omega_2(t) = 1/2! \int_0^t dt_1 \int_0^{t_1} dt_2[A(t_1), A(t_2)] $, \
$ \Omega_3(t) = 1/3! \int_0^t dt_1 \int_0^{t_1} dt_2 \int_0^{t_2} dt_3([A(t_1),[A(t_2),A(t_3)]] + [A(t_3),[A(t_2),A(t_1)]]) $

$$U(T) = \lim\limits_{n\rightarrow\infin} \text{exp}[\displaystyle\sum_{k=1}^n\Omega_k]$$

The expression of this problem is handled internally by QCSOS.jl, so we proceed to define the Fixed Time Problem and hit solve.

In [5]:
ϵ = 1e-4
η_eps = 1e-2
η_lambda = 1e-4
p = 3
problem = QuantumUnitaryFixedTimeProblem(U_target, problem, deg, p)
@info "Optimization problem constructed successfully."

qcsos_solver = QCSOS_Solver(p, problem, ϵ, η_eps, η_lambda)
_, reduced_solvers = solve!(qcsos_solver)

[ Info: ### Executing Symmetry Reduction ###
[ Info: Polynomial function f has degree: 12
[ Info: Symmetric Group 4
[ Info: End of decompose, returning...
[ Info: Optimization problem constructed successfully.
[ Info: Performing RRQR regularization
[ Info: ### Executing face reduction ###
[ Info: Performing face reduction on cone 1 of 5
[ Info: Subproblem constructed
[ Info: Iter. |      Dual Res. |    Primal Res. |     Cent. Res. |      Dual. Gap |    Primal Obj. |      Dual Obj. |      Step size |            b'y |
[ Info:     6 |     0.00741229 |    9.85392e-17 |       0.012817 |     0.00141561 |            0.0 |     0.00189381 |       0.696795 |    -0.00189381 |
[ Info: WARNING: gap s'z -0.0018938087842471248 above threshold 0.0001
[ Info: Dimension of cone objectid=14339039430940131572 reduced from 378 to 91
[ Info: Reduced problem, i = 1


[ Info: Optimize called
[ Info:                Solver parameters 


[ Info:     4 |      0.0301056 |     0.00932618 |      0.0677909 |      0.0118556 |            0.0 |     -0.0447675 |       0.623845 |      0.0447675 |
[ Info: Face reduced to n = 13
[ Info: Subproblem reduced


[ Info:                Objective limit:  -Inf
[ Info:                 Limit solution:  0.0
[ Info:                 Max iterations:  200
[ Info:                Num constraints:  243
[ Info:                    Num threads:  1
[ Info:               Preferred device:  CPU
[ Info: Iterative Refinement Max Iterations:  1
[ Info: Iterative Refinement Trigger Mode:  NO_ITERATIVE_REFINEMENT
[ Info:  Presolve Regularization Method: none
[ Info: Presolve Scaling (Equilibration) Method: ruiz
[ Info:       System Solver KKT method:  QR and Cholesky
[ Info:           Time limit (seconds):  1000000
[ Info:         Tolerance gap absolute:  0.01
[ Info:         Tolerance gap relative:  0.01
[ Info:           Tolerance optimality:  0.01
[ Info: Performing Ruiz Equilibration
[ Info: Solving for primal dual starting points
[ Info: Executing main optimization loop
[ Info: Iter. |      Dual Res. |    Primal Res. |     Cent. Res. |      Dual. Gap |    Primal Obj. |      Dual Obj. |      Step size |
[ Info:  

[ Info: Reproject iterate
[ Info: Reproject iterate
[ Info: Reproject iterate
[ Info: Face reduction unsuccessful, REDUCED_PROBLEM_UNSOLVED


(Any[], Any[])

# Evaluating the solution
To evaluate the solution we first obtain the control coefficients. From this we can then determine the infidelity achieved by comparing the target unitary with the estimated unitary returned by the solver.

In [ ]:
# get solution
solution = get_solution(problem.program, qcsos_solver.x_vec)

# evaluate solution
coefficients = get_control(problem, solution)
@info("Extracted control coefficients")
H_result = get_unitary_from_control(problem, coefficients)
infidelity = get_infidelity(U_target, H_result)
@info("Infidelity: $(infidelity)")

┌ Info: Numerical rank of solution (ideally 1) with tolerance 0.0001 is 5
└ @ QCSOS /home/alexander/Documents/alexander_leong/projects/QCSOS.jl/src/solver.jl:74
┌ Info: Extracted control coefficients
└ @ Main /home/alexander/Documents/alexander_leong/projects/QCSOS.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sdnNjb2RlLXJlbW90ZQ==.jl:6
┌ Info: Infidelity: 0.10189508452900675
└ @ Main /home/alexander/Documents/alexander_leong/projects/QCSOS.jl/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sdnNjb2RlLXJlbW90ZQ==.jl:9
